# Target Encoding Experiment Results

## Objective

Evaluate the impact of replacing One-Hot Encoding (OHE) with Target Encoding for different categorical feature combinations while keeping the remaining categorical features One-Hot Encoded.

## Top 10 Results

| Rank | Target Encoded Feature(s) | Best Model | Test AP | Train AP | Validation AP |
|----:|----------------------------|------------|------:|---------:|--------------:|
| 1 | Destination | LogisticRegression (Elastic Net) | 0.105805 | 0.084846 | **0.084918** |
| 2 | Destination | LogisticRegression (Base) | 0.104602 | 0.085498 | 0.084868 |
| 3 | Destination | LogisticRegression (Ridge) | 0.104950 | 0.085080 | 0.084438 |
| 4 | Destination | LogisticRegression (Lasso) | 0.106088 | 0.084328 | 0.084429 |
| 5 | Destination, Product Name | LogisticRegression (Lasso) | **0.111676** | 0.078908 | 0.084036 |
| 6 | Destination, Agency | LogisticRegression (Lasso) | 0.106347 | 0.083455 | 0.083970 |
| 7 | None (All OHE) | LogisticRegression (Base) | 0.104564 | 0.087203 | 0.083697 |
| 8 | None (All OHE) | LogisticRegression (Elastic Net) | 0.105818 | 0.087605 | 0.083485 |
| 9 | Agency, Product Name | GradientBoost | 0.102060 | 0.176639 | 0.083405 |
| 10 | None (All OHE) | LogisticRegression (Lasso) | 0.106671 | 0.086969 | 0.083256 |

## Conclusion

- **Destination** consistently appeared in the highest-performing experiments, indicating that it benefits the most from Target Encoding.
- The best validation performance (**0.084918**) was achieved by applying **Target Encoding only to Destination** with **Logistic Regression (Elastic Net)**.
- Although **Destination + Product Name** produced the highest Test Average Precision, its validation performance was lower than using **Destination** alone, suggesting mild overfitting.
- The baseline configuration (**all One-Hot Encoding**) remained competitive, indicating that Target Encoding provides only a modest improvement overall.
- Based on these results, the final preprocessing applies **Target Encoding to Destination** while keeping all remaining categorical features **One-Hot Encoded**.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

In [3]:
READ_CSV="../../data/interim/data_travel_insurance_interim.csv"
RANDOM_STATE=42
SAVE_RESULT = "results/feature_encoding_encoding_optimization_results.csv"

In [4]:
from src.utils import benchmark_models

from itertools import combinations

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

from feature_engine.outliers import Winsorizer

In [5]:
df = pd.read_csv(READ_CSV)
df.head()

,Agency,Agency Type,Distribution Channel,Product Name,Duration,Destination,Net Sales,Commission,Age,Claim,Is Refund,Suspected Fraud,Commission Rate
0,C2B,Airlines,Online,Annual Silver Plan,365,SINGAPORE,216.0,54.0,57,0,No,No,0.25
1,EPX,Travel Agency,Online,Cancellation Plan,4,MALAYSIA,10.0,0.0,33,0,No,No,0.00
2,JZI,Airlines,Online,Basic Plan,19,INDIA,22.0,7.7,26,0,No,No,0.35
3,EPX,Travel Agency,Online,2 way Comprehensive Plan,20,UNITED STATES,112.0,0.0,59,0,No,No,0.00
4,C2B,Airlines,Online,Bronze Plan,8,SINGAPORE,16.0,4.0,28,0,No,No,0.25


In [6]:
x = df.drop(columns=["Claim"])
y = df["Claim"]


In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [8]:
NUMERIC_COLS = [features for features in x_train.columns if x_train[features].dtypes != "O"]
CATEGORICAL_COLS = [features for features in x_train.columns if x_train[features].dtypes == "O"]


In [9]:
numeric_pipeline = Pipeline([
     ("winsorizer_iqr", Winsorizer(capping_method="iqr", fold=1.5)),
    ("RobustScaler", RobustScaler()),
])

categorical_ohe_pipeline = Pipeline([
     ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

categorical_target_pipeline = Pipeline([
     ("TargetEncoder", TargetEncoder())
])

In [10]:
logreg_base = LogisticRegression(random_state=RANDOM_STATE)
logreg_lasso = LogisticRegression(penalty="l1", solver="liblinear", random_state=RANDOM_STATE)
logreg_ridge = LogisticRegression(penalty="l2", solver="saga", random_state=RANDOM_STATE)
logreg_elasticnet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, random_state=RANDOM_STATE)
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(random_state=RANDOM_STATE)
ab = AdaBoostClassifier(random_state=RANDOM_STATE)
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
xgb = XGBClassifier(random_state=RANDOM_STATE)


In [ ]:
models = {
    "LogisticRegressionBase": logreg_base,
    "LogisticRegressionLasso": logreg_lasso,
    "LogisticRegressionRidge": logreg_ridge,
    "LogisticRegressionElasticNet": logreg_elasticnet,
    "DecisionTree": dt,
    "KNearestNeigbor": knn,
    "RandomForest": rf,
    "AdaBoost": ab,
    "GradientBoost": gb,
    "XGBoost": xgb
}

target_candidates = [
    col
    for col in CATEGORICAL_COLS
    if x_train[col].nunique() >= 3
]

print("Target Encoding Candidates:", target_candidates)

target_cols = [[]] 

for r in range(1, len(target_candidates) + 1):
    target_cols.extend(combinations(target_candidates, r))

all_results = []

for te_cols in target_cols:

    te_cols = list(te_cols)

    ohe_cols = [col for col in CATEGORICAL_COLS if col not in te_cols]

    transformers = [
        ("numeric", numeric_pipeline, NUMERIC_COLS),
        ("ohe", categorical_ohe_pipeline, ohe_cols),
    ]

    if te_cols:
        transformers.append(
            ("target", categorical_target_pipeline, te_cols)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression())
    ])

    experiment_results = benchmark_models(
        pipeline,
        models,
        x_train,
        y_train,
        x_test,
        y_test,
        5,
        "Base"
    )

    experiment_results["TargetEncoded"] = (
        "None (All OHE)"
        if not te_cols
        else ", ".join(te_cols)
    )

    all_results.append(experiment_results)

results = pd.concat(all_results, ignore_index=True)

In [18]:
results = results.sort_values("mean_ap_validate_score", ascending=False)
results[0:10]

,name,ap_test_score,mean_ap_train_score,mean_ap_validate_score,f1_test_score,recall_test_score,precision_test_score,accuracy_test_score,TargetEncoded
30,LogisticRegressionElasticNet_Base,0.105805,0.084846,0.084918,0.0,0.0,0.0,0.982820,Destination
31,LogisticRegressionBase_Base,0.104602,0.085498,0.084868,0.0,0.0,0.0,0.982820,Destination
32,LogisticRegressionRidge_Base,0.104950,0.085080,0.084438,0.0,0.0,0.0,0.982820,Destination
33,LogisticRegressionLasso_Base,0.106088,0.084328,0.084429,0.0,0.0,0.0,0.982820,Destination
60,LogisticRegressionLasso_Base,0.111676,0.078908,0.084036,0.0,0.0,0.0,0.982820,"Product Name, Destination"
50,LogisticRegressionLasso_Base,0.106347,0.083455,0.083970,0.0,0.0,0.0,0.982820,"Agency, Destination"
0,LogisticRegressionBase_Base,0.104564,0.087203,0.083697,0.0,0.0,0.0,0.982820,None (All OHE)
1,LogisticRegressionElasticNet_Base,0.105818,0.087605,0.083485,0.0,0.0,0.0,0.982820,None (All OHE)
40,GradientBoost_Base,0.102060,0.176639,0.083405,0.0,0.0,0.0,0.982057,"Agency, Product Name"
2,LogisticRegressionLasso_Base,0.106671,0.086969,0.083256,0.0,0.0,0.0,0.982820,None (All OHE)


In [19]:
results.to_csv(SAVE_RESULT, index=False)